<a href="https://colab.research.google.com/github/basmalam654-lab/Data-Mining/blob/main/Data_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTS AND LIBS


In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
import time
!pip install pyclustering
from pyclustering.cluster.kmedoids import kmedoids
from pyclustering.utils import distance_metric, type_metric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 22.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyclustering: filename=pyclustering-0.10.1.2-py3-none-any.whl size=2395100 sha256=5bda2107f0e8ab4cd01d2c940354f26079285d643fade23f01a0d232830a8a9b
  Stored in directory: /root/.cache/pip/wheels/68/29/b4/131bd7deec3663cc311ab9aa64d6517c3e3ec24bcadfc32f74
Successfully built pyclustering


In [ ]:
df = pd.read_csv("/content/train.csv (1).zip", on_bad_lines='warn')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.csv (1).zip'

In [ ]:
df.shape

In [ ]:
df.tail()

In [ ]:
df.at[df.index[-1], 'X'] =-122.405469
df.at[df.index[-1], 'Y'] =37.79387
df.tail()

In [ ]:
df.info()

In [ ]:
df['Dates'] = pd.to_datetime(df['Dates'])

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [ ]:
df.shape

In [ ]:
df[df['Address'] == '700 Block of COMMERCIAL ST']

In [ ]:
df.describe()

In [ ]:
df.nunique()

In [ ]:
df["Category"].unique()

In [ ]:
df["PdDistrict"].unique()

In [ ]:
df["Resolution"].unique()

In [ ]:
# Total number of records for each crime category
g1 = df.groupby("Category").size()
g1

In [ ]:
# Total number of crimes per police district.
g2 = df.groupby("PdDistrict").size()
g2

In [ ]:
#Number of cases per resolution type
g3 = df.groupby("Resolution").size()
g3

In [ ]:
#Number of crimes reported at each address.
g4 = df.groupby("Address").size()
g4

In [ ]:
# Extract relevant numerical features (geographic coordinates)
features = ['X', 'Y']  # Longitude and latitude
X = df[features]

In [ ]:
# Step 2: Remove outliers using IQR

x = df['X']
y = df['Y']


Q1_X = df['X'].quantile(0.25)
Q3_X = df['X'].quantile(0.75)
IQR_X = Q3_X - Q1_X

Q1_Y = df['Y'].quantile(0.25)
Q3_Y = df['Y'].quantile(0.75)
IQR_Y = Q3_Y - Q1_Y

lower_bound_X = Q1_X - 1.5 * IQR_X
upper_bound_X = Q3_X + 1.5 * IQR_X

lower_bound_Y = Q1_Y - 1.5 * IQR_Y
upper_bound_Y = Q3_Y + 1.5 * IQR_Y

df_cleaned = df[(df['X'] >= lower_bound_X) & (df['X'] <= upper_bound_X) &
                (df['Y'] >= lower_bound_Y) & (df['Y'] <= upper_bound_Y)]

outliers = df[
    (x < (Q1_X - 1.5 * IQR_X)) |(x > (Q3_X + 1.5 * IQR_X)) |
    (y < (Q1_Y - 1.5 * IQR_Y)) | (y > (Q3_Y + 1.5 * IQR_Y))
]


print("number of Outliers:", len(outliers))
display(outliers)


In [ ]:
# Step 3: Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cleaned[features])

In [ ]:
# Step 4: Sample the data
sample_size = 10000
np.random.seed(42)
sample_indices = np.random.choice(X_scaled.shape[0], sample_size, replace=False)
X_sample = X_scaled[sample_indices]

In [ ]:
# Step 5: Apply k-Medoids clustering
num_clusters = 10  # Define the number of clusters

# Randomly select initial medoids
initial_medoids = np.random.choice(range(X_sample.shape[0]), num_clusters, replace=False)

# Initialize and run k-Medoids
kmedoids_instance = kmedoids(X_sample, initial_medoids, metric=distance_metric(type_metric.EUCLIDEAN))
kmedoids_instance.process()

# Get clusters and final medoids
clusters = kmedoids_instance.get_clusters()
final_medoids = kmedoids_instance.get_medoids()

print("Clusters:", clusters)
print("Final Medoids:", final_medoids)

In [ ]:
# Step 6: Analyze and visualize the results
# Prepare data for visualization
cluster_labels = np.zeros(X_sample.shape[0])
for cluster_idx, cluster in enumerate(clusters):
    cluster_labels[cluster] = cluster_idx

# Visualize clusters
plt.figure(figsize=(10, 8))
plt.scatter(X_sample[:, 0], X_sample[:, 1], c=cluster_labels, cmap='viridis', s=1, label='Data Points')
plt.scatter(X_sample[final_medoids, 0], X_sample[final_medoids, 1], c='red', marker='x', s=100, label='Medoids')
plt.title("Crime Clusters (k-Medoids)")
plt.xlabel("Longitude (scaled)")
plt.ylabel("Latitude (scaled)")
plt.legend()
plt.show()

In [ ]:
dff=df.copy()

In [ ]:
#convert date in date time
dff["Datetime"] = pd.to_datetime(dff["Dates"])

In [ ]:
# drop unnecessary columns
dff.drop(["Dates"], axis=1, inplace=True)

# dff.rename(columns={"ComputedDay": "DayOfWeek"}, inplace=True)
dff.head()

In [ ]:
dff = df.drop(outliers.index)
print("عدد البيانات بعد حذف الـ Outliers:", len(dff))
display(dff.head())

In [ ]:
dff.nunique()

In [ ]:
dff["Category"].unique()

In [ ]:
dff["PdDistrict"].unique()

In [ ]:
dff["Resolution"].unique()

In [ ]:
# Total number of records for each crime category
g1 = dff.groupby("Category").size()
g1

In [ ]:
# Total number of crimes per police district.
g2 = dff.groupby("PdDistrict").size()
g2

In [ ]:
#Number of cases per resolution type
g3 = dff.groupby("Resolution").size()
g3

In [ ]:
#Number of crimes reported at each address.
g4 = dff.groupby("Address").size()
g4

In [ ]:
# Dates can be casted from object into datetime or numbers
dff.Dates = pd.to_datetime(dff.Dates)

first_date = dff.Dates.min()
last_date = dff.Dates.max()


print(f"  First Date of Crime: {first_date}")
print(f"  Last Date of Crime: {last_date}")
print(f"  Total Years of Crime: {(last_date-first_date).days // 365}")

In [ ]:
dff['Dates'] = pd.to_datetime(df['Dates'])


dff['Year'] = df['Dates'].dt.year

crimes_per_year = dff['Year'].value_counts().sort_index()

print(crimes_per_year)

In [ ]:
colors = plt.cm.tab20.colors

crimes_per_year.plot(kind='bar',
                     color=colors[:len(crimes_per_year)],
                     title='Number of Crimes by Year')

plt.xlabel('Year')
plt.ylabel('Number of Crimes')
plt.show()

In [ ]:
dff['Dates'] = pd.to_datetime(df['Dates'])


dff['Hour'] = dff['Dates'].dt.hour

crimes_per_hour = dff['Hour'].value_counts().sort_index()

colors = plt.cm.Set3.colors

crimes_per_hour.plot(kind='bar',
                     color=colors[:len(crimes_per_hour)],
                     title='Number of Crimes by Hour')

plt.xlabel('Hour of Day')
plt.ylabel('Number of Crimes')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Bar Chart - Number of Crimes per Category
plt.figure(figsize=(12,6))
df['Category'].value_counts().plot(kind='bar', color='skyblue')
plt.title('Number of Crimes per Category')
plt.xlabel('Crime Category')
plt.ylabel('Number of Crimes')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Pie Chart - Crime Distribution by Day of the Week
day_counts = df['DayOfWeek'].value_counts()
plt.figure(figsize=(7,7))
plt.pie(day_counts, labels=day_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Crime Distribution by Day of the Week')
plt.show()

In [ ]:
# 3. Box Plot - Longitude Distribution by Police District
plt.figure(figsize=(12,6))
sns.boxplot(x='PdDistrict', y='X', data=df)
plt.title('Longitude Distribution of Crimes by Police District')
plt.xlabel('Police District')
plt.ylabel('Longitude')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Scatter Plot - Geographical Crime Locations
plt.figure(figsize=(10,8))
plt.scatter(df['X'], df['Y'], alpha=0.2, s=10, color='red')
plt.title('Geographical Distribution of Crimes')
plt.xlabel('Longitude (X)')
plt.ylabel('Latitude (Y)')
plt.grid(True)
plt.show()

In [ ]:
# 5. Heatmap - Crime Distribution by Day and Hour
dff["Datetime"] = pd.to_datetime(dff["Dates"])
dff.loc[: ,'Hour'] = dff['Datetime'].dt.hour
heat_data = pd.crosstab(dff['DayOfWeek'], dff['Hour'])
# Reorder days of the week
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heat_data = heat_data.reindex(days_order)
plt.figure(figsize=(25,10))
sns.heatmap(heat_data ,annot=True,cmap='YlOrRd')
plt.title('Heatmap of Crimes by Day and Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Day of the Week')
plt.show()

In [ ]:
for data in [dff]:
    data['Dates'] = pd.to_datetime(data['Dates'])
    data['Hour'] = data['Dates'].dt.hour
    data['Month'] = data['Dates'].dt.month
    data['Year'] = data['Dates'].dt.year


le_day = LabelEncoder()
le_district = LabelEncoder()
le_category = LabelEncoder()

dff['DayOfWeek'] = le_day.fit_transform(dff['DayOfWeek'])
dff['PdDistrict'] = le_district.fit_transform(dff['PdDistrict'])
dff['Category'] = le_category.fit_transform(dff['Category'])


In [ ]:
from sklearn.model_selection import train_test_split
feature_cols = ['DayOfWeek', 'PdDistrict', 'X', 'Y', 'Hour', 'Month', 'Year']
X = dff[feature_cols]
y = dff['Category']
vectorizer = TfidfVectorizer(max_features=100)
desc_features = vectorizer.fit_transform(dff['Descript']).toarray()


X = np.hstack((X, desc_features))


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_val_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_val_pred)
print(f" Accuracy: {accuracy:.2f}")

In [ ]:
# Reduce the number of trees and depth
# Optimized Random Forest implementation
model = RandomForestClassifier(
    n_estimators=30,
    max_depth=8,
    min_samples_split=50,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
print("Starting model training...")
start_time = time.time()

model.fit(X_train, y_train)

end_time = time.time()
print(f"Training completed in {end_time - start_time:.2f} seconds")

# Evaluation
y_val_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred, average='weighted')
recall = recall_score(y_val, y_val_pred, average='weighted')
f1 = f1_score(y_val, y_val_pred, average='weighted')

print(f"\nAccuracy: {accuracy:f}")
print(f"Precision: {precision:f}")
print(f"Recall: {recall:f}")
print(f"F1 Score: {f1:f}")

In [ ]:
# 4. Confusion matrix visualization
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_val, y_val_pred)

# show only top N classes
top_n = 15  # Adjust based on your needs
if len(le_category.classes_) > top_n:
    # Get indices of top N most frequent classes
    class_counts = np.sum(cm, axis=1)
    top_classes = np.argsort(-class_counts)[:top_n]
    cm = cm[top_classes][:, top_classes]
    class_names = le_category.classes_[top_classes]
else:
    class_names = le_category.classes_

# Plot the confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', cbar=False,
            xticklabels=class_names,
            yticklabels=class_names)
plt.title(f'Confusion Matrix (Top {len(class_names)} Classes)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()